<a href="https://colab.research.google.com/github/lipchenko/machine_learning_lipchenko/blob/main/my_trees_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: cравнение древовидных алгоритмов на NLP-данных

**Задача:** oбучить разные древовидные модели на текстах новостей, замерить test accuracy и время обучения, заполнить таблицы.

## 1. Загрузка и подготовка данных (код дан)

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time
import pandas as pd

# Загружаем 3 категории
categories = ['rec.sport.baseball', 'sci.space', 'comp.graphics']
newsgroups = fetch_20newsgroups(subset='all', categories=categories,
                                shuffle=True, random_state=42)

# TF-IDF векторизация
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                             stop_words='english')
X = vectorizer.fit_transform(newsgroups.data)
y = newsgroups.target

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (2363, 5000)
Размер тестовой выборки: (591, 5000)


In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00


## 2. Образец: Дерево решений (Decision Tree)

**Гиперпараметр:** `max_depth`

In [3]:
from sklearn.tree import DecisionTreeClassifier

results_tree = []

# Образец: мы делаем перебор значений max_depth = [3, 5, 10, 20, None])
# Для каждого: замер времени, обучение, accuracy на тесте, сохранение в results_tree

for depth in [3, 5, 10, 20, None]:
    start = time.time()
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_tree.append({'max_depth': depth,
                         'test_accuracy': round(acc, 4),
                         'time_sec': round(fit_time, 2)})
    print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_tree = pd.DataFrame(results_tree)
print("\nТаблица 1. Дерево решений")
print(df_tree.to_markdown(index=False))

depth=3: acc=0.6244, time=0.12s
depth=5: acc=0.6971, time=0.14s
depth=10: acc=0.7716, time=0.28s
depth=20: acc=0.8409, time=0.38s
depth=None: acc=0.8697, time=0.72s

Таблица 1. Дерево решений
|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           3 |          0.6244 |       0.12 |
|           5 |          0.6971 |       0.14 |
|          10 |          0.7716 |       0.28 |
|          20 |          0.8409 |       0.38 |
|         nan |          0.8697 |       0.72 |


In [4]:
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'{train_acc:.4f}')
print(f'{test_acc:.4f}')

1.0000
0.8697


## 3. Заполните таблицы для остальных алгоритмов

По аналогии с образцом обучите следующие модели и запишите результаты.

### 3.1. Случайный лес (Random Forest)

**Гиперпараметр:** `max_depth` (фиксируем `n_estimators=100`)

|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           5 |          0.9002 |       0.38 |
|          10 |          0.9154 |       0.65 |
|          20 |          0.9357 |       1.09 |
|         nan |          0.9526 |       2.05 |

In [5]:
from sklearn.ensemble import RandomForestClassifier

results_rf = []

# Ваш код здесь (перебор max_depth)
for depth in [5, 10, 20, None]:
    start = time.time()
    clf = RandomForestClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_rf.append({'max_depth': depth, 'test_accuracy': round(acc, 4),
                       'time_sec': round(fit_time, 2)})
    print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_rf = pd.DataFrame(results_rf)
print("\nТаблица 2. Случайный лес")
print(df_rf.to_markdown(index=False))

depth=5: acc=0.9002, time=0.36s
depth=10: acc=0.9154, time=0.79s
depth=20: acc=0.9357, time=2.67s
depth=None: acc=0.9526, time=3.38s

Таблица 2. Случайный лес
|   max_depth |   test_accuracy |   time_sec |
|------------:|----------------:|-----------:|
|           5 |          0.9002 |       0.36 |
|          10 |          0.9154 |       0.79 |
|          20 |          0.9357 |       2.67 |
|         nan |          0.9526 |       3.38 |


In [6]:
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'{train_acc:.4f}')
print(f'{test_acc:.4f}')

1.0000
0.9526


### 3.2. Градиентный бустинг (Gradient Boosting)

**Гиперпараметр:** `learning_rate` (фиксируем `n_estimators=100, max_depth=3`)

|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|            0.01 |          0.8782 |      36    |
|            0.05 |          0.9475 |      35.83 |
|            0.1  |          0.9577 |      36.2  |
|            0.5  |          0.9662 |      35.35 |

In [7]:
from sklearn.ensemble import GradientBoostingClassifier

results_gb = []

for learning_rate in [0.01, 0.05, 0.1, 0.5]:
    start = time.time()
    clf = GradientBoostingClassifier(learning_rate=learning_rate, max_depth=3,
                                     n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_gb.append({'learning_rate': learning_rate,
                       'test_accuracy': round(acc, 4),
                       'time_sec': round(fit_time, 2)})
    print(f"learning_rate={learning_rate}: acc={acc:.4f}, time={fit_time:.2f}s")

df_gb = pd.DataFrame(results_gb)
print("\nТаблица 3. Градиентный бустинг")
print(df_gb.to_markdown(index=False))

learning_rate=0.01: acc=0.8782, time=31.09s
learning_rate=0.05: acc=0.9475, time=21.38s
learning_rate=0.1: acc=0.9577, time=21.71s
learning_rate=0.5: acc=0.9662, time=22.57s

Таблица 3. Градиентный бустинг
|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|            0.01 |          0.8782 |      31.09 |
|            0.05 |          0.9475 |      21.38 |
|            0.1  |          0.9577 |      21.71 |
|            0.5  |          0.9662 |      22.57 |


In [8]:
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'{train_acc:.4f}')
print(f'{test_acc:.4f}')

1.0000
0.9662


### 3.3. XGBoost

**Гиперпараметр:** `gamma` (фиксируем `n_estimators=100, learning_rate=0.1`)

|   gamma |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|     0   |          0.9475 |      22.02 |
|     0.1 |          0.9492 |      21.67 |
|     0.5 |          0.9526 |      15.1  |
|     1   |          0.9492 |      12.73 |

In [9]:
from xgboost import XGBClassifier

results_xgb = []

for gamma in [0, 0.1, 0.5, 1.0]:
    start = time.time()
    clf = XGBClassifier(gamma=gamma, use_label_encoder=False,
                        eval_metric='logloss', random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_xgb.append({'gamma': gamma,
                        'test_accuracy': round(acc, 4),
                        'time_sec': round(fit_time, 2)})
    print(f"gamma={gamma}: acc={acc:.4f}, time={fit_time:.2f}s")

df_xgb = pd.DataFrame(results_xgb)
print("\nТаблица 4. XGBoost")
print(df_xgb.to_markdown(index=False))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:07:18] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


gamma=0: acc=0.9475, time=15.47s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:07:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


gamma=0.1: acc=0.9492, time=13.25s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:07:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


gamma=0.5: acc=0.9526, time=8.86s


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:07:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


gamma=1.0: acc=0.9492, time=7.95s

Таблица 4. XGBoost
|   gamma |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|     0   |          0.9475 |      15.47 |
|     0.1 |          0.9492 |      13.25 |
|     0.5 |          0.9526 |       8.86 |
|     1   |          0.9492 |       7.95 |


In [10]:
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'{train_acc:.4f}')
print(f'{test_acc:.4f}')

0.9970
0.9492


### 3.4. AdaBoost

**Гиперпараметр:** `learning_rate` (фиксируем `n_estimators=100`)

|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|             0.5 |          0.8629 |       6.03 |
|             1   |          0.9205 |       5.29 |
|             1.5 |          0.9255 |       5.97 |
|             2   |          0.8731 |       5.27 |

In [11]:
from sklearn.ensemble import AdaBoostClassifier

results_ada = []

for learning_rate in [0.5, 1.0, 1.5, 2.0]:
    start = time.time()
    clf = AdaBoostClassifier(n_estimators=100, learning_rate=learning_rate, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_ada.append({'learning_rate': learning_rate,
                        'test_accuracy': round(acc, 4),
                        'time_sec': round(fit_time, 2)})
    print(f"learning_rate={learning_rate}: acc={acc:.4f}, time={fit_time:.2f}s")

df_ada = pd.DataFrame(results_ada)
print("\nТаблица 5. AdaBoost")
print(df_ada.to_markdown(index=False))

learning_rate=0.5: acc=0.8629, time=3.79s
learning_rate=1.0: acc=0.9205, time=3.92s
learning_rate=1.5: acc=0.9255, time=3.67s
learning_rate=2.0: acc=0.8731, time=3.95s

Таблица 5. AdaBoost
|   learning_rate |   test_accuracy |   time_sec |
|----------------:|----------------:|-----------:|
|             0.5 |          0.8629 |       3.79 |
|             1   |          0.9205 |       3.92 |
|             1.5 |          0.9255 |       3.67 |
|             2   |          0.8731 |       3.95 |


In [12]:
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'{train_acc:.4f}')
print(f'{test_acc:.4f}')

0.9259
0.8731


### 3.5. LightGBM

**Гиперпараметр:** `num_leaves` (фиксируем `n_estimators=100, learning_rate=0.1`)

|   num_leaves |   test_accuracy |   time_sec |
|-------------:|----------------:|-----------:|
|           15 |          0.9662 |       6.3  |
|           31 |          0.9594 |       9.73 |
|           63 |          0.9577 |      13.87 |
|          127 |          0.9594 |      19.1  |

In [13]:
from lightgbm import LGBMClassifier

results_lgbm = []

for num_leaves in [15, 31, 63, 127]:
    start = time.time()
    clf = LGBMClassifier(num_leaves=num_leaves, n_jobs=-1, random_state=42)
    clf.fit(X_train, y_train)
    fit_time = time.time() - start
    acc = accuracy_score(y_test, clf.predict(X_test))
    results_lgbm.append({'num_leaves': num_leaves,
                         'test_accuracy': round(acc, 4),
                         'time_sec': round(fit_time, 2)})
    print(f"num_leaves={num_leaves}: acc={acc:.4f}, time={fit_time:.2f}s")

df_lgbm = pd.DataFrame(results_lgbm)
print("\nТаблица 6. LightGBM")
print(df_lgbm.to_markdown(index=False))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061918 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61063
[LightGBM] [Info] Number of data points in the train set: 2363, number of used features: 2963
[LightGBM] [Info] Start training from score -1.119999
[LightGBM] [Info] Start training from score -1.093126
[LightGBM] [Info] Start training from score -1.083076


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=15: acc=0.9662, time=2.82s
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061843 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61063
[LightGBM] [Info] Number of data points in the train set: 2363, number of used features: 2963
[LightGBM] [Info] Start training from score -1.119999
[LightGBM] [Info] Start training from score -1.093126
[LightGBM] [Info] Start training from score -1.083076


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=31: acc=0.9594, time=5.72s
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.058861 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 61063
[LightGBM] [Info] Number of data points in the train set: 2363, number of used features: 2963
[LightGBM] [Info] Start training from score -1.119999
[LightGBM] [Info] Start training from score -1.093126
[LightGBM] [Info] Start training from score -1.083076
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


num_leaves=63: acc=0.9577, time=7.07s
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.064891 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 61063
[LightGBM] [Info] Number of data points in the train set: 2363, number of used features: 2963
[LightGBM] [Info] Start training from score -1.119999
[LightGBM] [Info] Start training from score -1.093126
[LightGBM] [Info] Start training from score -1.083076
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
#LightGbM не хотел сравнивать тестовую и тренировочную accuracies без параметра num_leaves
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f"num_leaves={num_leaves}, train_acc={train_acc:.4f}, test_acc={test_acc:.4f}")

num_leaves=127, train_acc=1.0000, test_acc=0.9594


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### 3.6. CatBoost

**Гиперпараметр:** `depth` (фиксируем `iterations=100, learning_rate=0.1`)

|   depth |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|       3 |          0.9492 |      12.16 |
|       5 |          0.9526 |      31.78 |
|       7 |          0.9526 |     101.47 |
|      10 |          0.956  |     748.95 |

In [15]:
#Переделала данные в плотный массив, потому что CatBoost долго грузил и делал ошибки на TF-IDF
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

In [16]:
from catboost import CatBoostClassifier

results_cat = []

for depth in [3, 5, 7, 10]:
    start = time.time()
    clf = CatBoostClassifier(depth=depth, iterations=100,
                             verbose=50, random_state=42, thread_count=-1)
    clf.fit(X_train_dense, y_train)
    fit_time = time.time() - start
    y_pred = clf.predict(X_test_dense)
    acc = accuracy_score(y_test, y_pred)
    results_cat.append({'depth': depth, 'test_accuracy': round(acc, 4),
                        'time_sec': round(fit_time, 2)})
    print(f"depth={depth}: acc={acc:.4f}, time={fit_time:.2f}s")

df_cat = pd.DataFrame(results_cat)
print("\nТаблица 7. CatBoost")
print(df_cat.to_markdown(index=False))

Learning rate set to 0.5
0:	learn: 0.8808731	total: 272ms	remaining: 27s
50:	learn: 0.2331643	total: 4.83s	remaining: 4.64s
99:	learn: 0.1788203	total: 8.46s	remaining: 0us
depth=3: acc=0.9492, time=9.23s
Learning rate set to 0.5
0:	learn: 0.8400777	total: 686ms	remaining: 1m 7s
50:	learn: 0.1930887	total: 11s	remaining: 10.5s
99:	learn: 0.1301336	total: 21s	remaining: 0us
depth=5: acc=0.9526, time=21.60s
Learning rate set to 0.5
0:	learn: 0.8380700	total: 1.74s	remaining: 2m 52s
50:	learn: 0.1608495	total: 33.6s	remaining: 32.3s
99:	learn: 0.1078483	total: 1m 4s	remaining: 0us
depth=7: acc=0.9526, time=65.03s
Learning rate set to 0.5
0:	learn: 0.8241805	total: 4.87s	remaining: 8m 2s
50:	learn: 0.1318827	total: 3m 48s	remaining: 3m 39s
99:	learn: 0.0789386	total: 7m 31s	remaining: 0us
depth=10: acc=0.9560, time=452.66s

Таблица 7. CatBoost
|   depth |   test_accuracy |   time_sec |
|--------:|----------------:|-----------:|
|       3 |          0.9492 |       9.23 |
|       5 |        

In [17]:
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc = accuracy_score(y_test, clf.predict(X_test))
print(f'{train_acc:.4f}')
print(f'{test_acc:.4f}')

1.0000
0.9560


## 4. Итоговая таблица лучших результатов

In [18]:
# Соберите лучшие результаты из каждой таблицы вручную или автоматически
# Пример структуры:

summary = pd.DataFrame([
    ('Decision Tree', 20, 0.8697, 0.62),  # замените 0 на ваши значения
    ('Random Forest', None, 0.9526, 2.05),
    ('Gradient Boosting', 0.5, 0.9662, 35.35),
    ('XGBoost', 0.5, 0.9526, 15.1),
    ('AdaBoost', 1.5, 0.9255, 5.97),
    ('LightGBM', 15, 0.9662, 6.3),
    ('CatBoost', 10, 0.956, 748.95),
], columns=['Algorithm', 'Best params', 'Best test acc', 'Time (sec)'])

print("Итоговая таблица лучших результатов")
print(summary.to_markdown(index=False))

Итоговая таблица лучших результатов
| Algorithm         |   Best params |   Best test acc |   Time (sec) |
|:------------------|--------------:|----------------:|-------------:|
| Decision Tree     |          20   |          0.8697 |         0.62 |
| Random Forest     |         nan   |          0.9526 |         2.05 |
| Gradient Boosting |           0.5 |          0.9662 |        35.35 |
| XGBoost           |           0.5 |          0.9526 |        15.1  |
| AdaBoost          |           1.5 |          0.9255 |         5.97 |
| LightGBM          |          15   |          0.9662 |         6.3  |
| CatBoost          |          10   |          0.956  |       748.95 |


## 5. Вопросы (ответить текстом в следующей ячейке)

1. Какой алгоритм показал максимальную точность? Какие параметры к этому привели?

2. Какой алгоритм быстрее всего обучался? Во сколько раз он быстрее самого медленного?

3. У каких алгоритмов наблюдалось переобучение? При каких параметрах?

4. Какой алгоритм вы выбрали бы для продакшн-системы с миллионом текстов? Почему?

# Напишите свои ответы здесь

**Ответ 1:**
Алгоритмы Gradient Boosting и LightGBM показали максимальный показатель 0.9662. Gradient Boosting показал максимальный показатель accuracy при learning rate = 0.5, а LightGBM при num leaves = 15. При этом LightGBM выполнил работу намного быстрее.


**Ответ 2:**
Decision Tree выполнил работу быстрее всех за 0 секунд. Он был на 748 секунд быстрее самого медленного алгоритма CatBoost.


**Ответ 3:**
сравним результаты тестовой и тренировочной accuracy. Decision Tree: test acuuracy - 0.869 и train accuracy - 1.0. Получается, у модели идет переобучение, так тестовая метрика на 20% отстает от тренировочной. Forest Random: train accuracy - 1.0, test accuracy - 0.952. Получается, модель немного переобучается. Gradient Boosting: train accuracy - 1.0, test accuracy - 0.966. Есть маленькое переобучение. XGBoost: train accuracy - 0.99, test accuracy - 0.95. Также маленькое переобучение. AdaBoost: train accuracy - 0.92, test accuracy - 0.87. Маленькое переобучение. LightGBM: train accuracy - 1.0, test accuracy - 0.95. Маленькое переобучение. CatBoost: train accuracy - 1.0, test accuracy - 0.95. Маленькое переобучение. Таким образом, все модели проходят переобучение.



**Ответ 4:**
LightGBM показывает бысокую точность и работает быстро. Также ему не нужны большие объемы памяти.

## Критерии оценки

| Что оценивается | Баллы |
|----------------|-------|
| Заполнены все 7 таблиц (правильно собраны accuracy и время) | 3 |
| Заполнена итоговая таблица лучших результатов | 1 |
| Ответы на 4 вопроса (по 0.5 балла) | 2 |
| Код воспроизводим (random_state=42, порядок ячеек корректен) | 1 |
| **Качество кода** (отсутствие дублирования, осмысленные имена переменных, циклы вместо копипасты, использование .to_markdown()) | 3 |
| **Итого** | **10** |

### Детали по качеству кода (+3 балла):
- **+1** — использование единого шаблона для всех экспериментов (цикл + list.append + pd.DataFrame)
- **+1** — правильное форматирование вывода (округление времени и accuracy до 2-4 знаков)
- **+1** — читаемые имена переменных, комментарии, отсутствие `eval()`, `exec()` и магических чисел

**Штрафы:**
- -1 балл, если код не запускается без ошибок
- -1 балл, если отсутствует `random_state=42` в любой из моделей
- -1 балл, если таблицы выведены криво (не через `to_markdown` или нечитаемый `print`)